# 04 · Baselines

Simple rules before any machine learning, so I know how much a model actually adds. Scores are MAE on normal loads (`MAE_clean`), in dollars.

## Baseline ladder
From one global rate per mile up to lane × equipment medians.

In [1]:
import sys; sys.path.append("../src")
import pandas as pd
from baselines import BASELINES
from cv import run_cv, run_hard

raw = pd.read_csv("../data/train_test.csv")

summary, per_fold = [], {}
for name, make in BASELINES.items():
    cv = run_cv(make, raw)
    hard = run_hard(make, raw)
    per_fold[name] = cv
    summary.append({
        "model": name,
        "MAE (cv mean)": cv["MAE"].mean(),
        "MAE_clean (cv mean)": cv["MAE_clean"].mean(),
        "MAE_clean fold 3": cv.loc[3, "MAE_clean"],
        "MAPE%_clean (cv mean)": cv["MAPE%_clean"].mean(),
        "MAE_clean hard subset": hard["MAE_clean"].mean(),
    })

display(pd.DataFrame(summary).set_index("model").round(2))
for name, cv in per_fold.items():
    print(name); display(cv.round(2))

,MAE (cv mean),MAE_clean (cv mean),MAE_clean fold 3,MAPE%_clean (cv mean),MAE_clean hard subset
model,,,,,
1 global rate/mile,249.25,197.56,203.69,9.37,226.87
2 by equipment,219.35,167.07,175.09,8.33,199.15
3 equipment x dist band,158.11,104.88,80.64,4.66,86.38
4 lane x equipment,156.36,103.02,77.37,4.45,86.38
5 linear regression,172.47,119.44,94.38,6.48,100.23


1 global rate/mile


,MAE,RMSE,MAPE%,MAE_clean,RMSE_clean,MAPE%_clean
fold,,,,,,
1,241.78,671.84,11.89,189.83,244.87,9.82
2,249.02,674.72,11.16,199.15,289.88,9.03
3,256.95,684.25,11.63,203.69,295.92,9.27


2 by equipment


,MAE,RMSE,MAPE%,MAE_clean,RMSE_clean,MAPE%_clean
fold,,,,,,
1,206.15,654.18,11.01,153.72,188.55,8.93
2,222.81,662.93,10.11,172.41,255.16,7.96
3,229.08,670.65,10.49,175.09,258.15,8.09


3 equipment x dist band


,MAE,RMSE,MAPE%,MAE_clean,RMSE_clean,MAPE%_clean
fold,,,,,,
1,201.46,665.12,8.32,148.91,198.15,6.22
2,136.61,630.02,6.17,85.10,116.50,3.94
3,136.27,638.22,6.32,80.64,108.54,3.81


4 lane x equipment


,MAE,RMSE,MAPE%,MAE_clean,RMSE_clean,MAPE%_clean
fold,,,,,,
1,200.53,665.25,8.25,147.87,197.30,6.13
2,135.46,631.39,6.00,83.82,116.63,3.75
3,133.08,638.67,5.97,77.37,107.30,3.46


5 linear regression


,MAE,RMSE,MAPE%,MAE_clean,RMSE_clean,MAPE%_clean
fold,,,,,,
1,218.07,668.37,10.09,165.73,210.06,7.95
2,149.61,633.00,7.99,98.23,129.57,5.77
3,149.73,641.85,8.24,94.38,126.01,5.73


## Where does the best baseline go wrong?
Checking its errors by month and what they correlate with. The bias tracks the market level: under-predicting in the May–Jun peak.

In [2]:
import sys; sys.path.append("../src")
import numpy as np, pandas as pd
from splits import make_folds, prepare_fold
from baselines import BASELINES

raw = pd.read_csv("../data/train_test.csv")

parts = []
for k, tr_idx, te_idx in make_folds(raw):
    train, test, _, _ = prepare_fold(raw, tr_idx, te_idx)
    m = BASELINES["3 equipment x dist band"]().fit(train)
    t = test[~test["is_outlier"]].copy()
    t["log_err"] = np.log(t["posted_rate"] / m.predict(t))   # >0 means we under-predicted
    t["fold"], t["month"], t["dow"] = k, t["date"].dt.to_period("M"), t["date"].dt.dayofweek
    parts.append(t)
allt = pd.concat(parts)

# 1. Is the error biased in time?
display(allt.groupby(["fold", "month"]).agg(
    bias=("log_err", "mean"), spread=("log_err", "std"), market_index=("market_index", "mean")).round(3))

# 2. What does the leftover error correlate with?
for k, d in allt.groupby("fold"):
    print(k, d[["log_err", "market_index", "quote_signal", "weight", "dow"]]
          .corr(method="spearman")["log_err"].drop("log_err").round(3).to_dict())

# 3. Are market_index / quote_signal date-level signals or row-level noise?
for c in ["market_index", "quote_signal"]:
    print(c, "| overall std:", round(raw[c].std(), 3),
          "| mean within-date std:", round(raw.groupby("date")[c].std().mean(), 3))

bias  spread  market_index
fold month                               
1    2025-05  0.048   0.045         1.301
     2025-06  0.072   0.047         1.280
2    2025-07  0.023   0.047         1.201
     2025-08 -0.008   0.046         0.981
3    2025-09  0.007   0.048         0.893
     2025-10  0.010   0.048         0.957

1 {'market_index': 0.132, 'quote_signal': 0.167, 'weight': 0.512, 'dow': -0.02}
2 {'market_index': 0.421, 'quote_signal': -0.231, 'weight': 0.499, 'dow': -0.067}
3 {'market_index': 0.149, 'quote_signal': 0.007, 'weight': 0.54, 'dow': 0.011}
market_index | overall std: 0.168 | mean within-date std: 0.025
quote_signal | overall std: 0.291 | mean within-date std: 0.261
